# E2: CMX + CMRF (Cross-Modal Residual Filtering) on PST900
**Team 41 · RGB-Thermal semantic segmentation · CMX → CMRF**

Same pipeline, dataset, training schedule (100 epochs) and robustness benchmark as **E1**. The **only** change is one lightweight module, **CMRF**, inserted at **encoder stage 3**, just before CMX's CM-FRM.

```
GPU check → Drive → install → PST900 → inspect data → clone + patch CMX → ADD CMRF → CMX-format data
→ MiT-B2 weights → config → smoke test → TRAIN (resumable) → save model → benchmark → E1 vs E2 → gate maps
```

**Before you start:** `Runtime → Change runtime type → GPU`. Everything persistent goes to Google Drive, so after a disconnect, Run all again and training resumes.

### What changes from E1 (and nothing else)
| | E1 | E2 |
|---|---|---|
| Model | CMX (MiT-B2) | CMX (MiT-B2) **+ CMRF at stage 3** |
| Loss | cross-entropy | cross-entropy **+ 0.1 × cross-modal prediction loss** |
| Extra cost | – | **+0.70% params, +0.71% GFLOPs** (measured) |
| Data, split, epochs, optimiser, LR, batch, crop, evaluation | – | **identical** |

**CMRF at the chosen stage:**
1. **Predict** thermal features from RGB features, and RGB features from thermal features (two small conv predictors).
2. **Residual** = actual − predicted, i.e. what one camera knows that the other **cannot explain**.
3. **Gate** (per pixel, per channel, values 0–1) decides how much of that residual is useful.
4. **Inject** the gated residual into the other stream; CMX's own CM-FRM and FFM then run unchanged.

The prediction loss uses **stop-gradient targets**, so the predictors learn real cross-modal correspondence and the features cannot collapse.

## 1. GPU check

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU, then re-run."
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. Google Drive (persistent storage) and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT   = '/content/drive/MyDrive/AIProj_CMX'      # everything persistent lives here
EXP       = 'E2_cmrf_s3_ep100'                             # change per experiment (E2_cmrf, ...)
LOG_DIR   = f'{PROJECT}/logs/{EXP}'                  # checkpoints + train logs
RESULTS   = f'{PROJECT}/results'                     # CSVs, figures
PRETRAIN  = f'{PROJECT}/pretrained'                  # MiT ImageNet weights (downloaded once)

DATA_DIR  = '/content/data'                          # local, fast; re-downloaded per session
PST_ROOT  = f'{DATA_DIR}/PST900_RGBT_Dataset'
CMX_DATA  = '/content/pst900_cmx'                    # PST900 in CMX folder layout (symlinks)
REPO      = '/content/RGBX_Semantic_Segmentation'

for d in (PROJECT, LOG_DIR, RESULTS, PRETRAIN, DATA_DIR):
    os.makedirs(d, exist_ok=True)
print('Project folder:', PROJECT)

## 3. Install dependencies
`time` is part of Python's standard library: putting it in `pip install` makes pip fail and **nothing** gets installed.

In [ ]:
!pip install -q gdown einops albumentations fvcore easydict timm tensorboardX opencv-python-headless

## 4. Download PST900

In [ ]:
import zipfile, shutil, glob

DRIVE_DS  = f'{PROJECT}/datasets'
FILE_ID   = '1hZeM-MvdUC_Btyok7mdF00RV-InbAadm'
LOCAL_ZIP = f'{DATA_DIR}/PST900_RGBT_Dataset.zip'
os.makedirs(DRIVE_DS, exist_ok=True)

if not os.path.exists(PST_ROOT):
    # 1) use a zip (or shortcut to it) already in your Drive
    zips = glob.glob(f'{DRIVE_DS}/*.zip')
    if zips:
        print('Copying from Drive:', zips[0])
        shutil.copy(zips[0], LOCAL_ZIP)
    # 2) otherwise try gdown (fails while Google's quota is exceeded)
    elif not os.path.exists(LOCAL_ZIP):
        !gdown {FILE_ID} -O {LOCAL_ZIP}

    assert os.path.exists(LOCAL_ZIP), (
        "Download blocked by Google's quota. Add a shortcut / upload the zip to "
        f"{DRIVE_DS} (see instructions), then re-run this cell.")

    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall(DATA_DIR)

    # keep a real copy in Drive so future sessions never depend on the public link
    if not glob.glob(f'{DRIVE_DS}/*.zip') or os.path.islink(glob.glob(f'{DRIVE_DS}/*.zip')[0]):
        shutil.copy(LOCAL_ZIP, f'{DRIVE_DS}/PST900_RGBT_Dataset_copy.zip')

print(sorted(os.listdir(DATA_DIR)))
for s in ('train', 'test'):
    print(s, sorted(os.listdir(f'{PST_ROOT}/{s}')), len(os.listdir(f'{PST_ROOT}/{s}/rgb')), 'images')

## 5. Toolkit files
Two helper modules are written to `/content`:
* **`rgbt_eval.py`**: dataset, corruptions, metrics, sliding-window evaluation, cost.
* **`cmx_colab.py`**: patches the official repo for Colab, converts PST900 to CMX layout, writes the config, loads checkpoints.

In [ ]:
%%writefile /content/rgbt_eval.py
"""
rgbt_eval.py -- Evaluation toolkit for the CMX / CMRF RGB-Thermal project (PST900)

1. PST900Dataset  : reads images exactly like the official CMX repo (OpenCV;
                    BGR channel order by default), labels at ORIGINAL resolution,
                    out-of-range labels -> 255 (ignored, never clipped into a class),
                    optional seeded corruption hook.
2. Corruptions    : fixed RGB / thermal degradations, 3 severities, global and LOCALIZED.
3. SegMetrics     : confusion-matrix mIoU, Dice, Sensitivity, Specificity
                    (per class / mean over all classes / mean over object classes).
4. evaluate(), run_benchmark(), robustness_summary()
                    sliding-window inference (like the CMX evaluator), clean + corrupted.
5. model_cost()   : params, GFLOPs (fvcore), latency, FPS.
"""

import os
import csv
import glob
import time
import random

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IGNORE_INDEX = 255

# Verify this order against the official PST900 README before reporting results.
PST900_CLASSES = ["Background", "Fire-Extinguisher", "Backpack", "Hand-Drill", "Survivor"]
PST900_PALETTE = [[0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 0]]  # RGB

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])


# =============================================================================
# 1. Corruptions (uint8 numpy; RGB image is HxWx3 in RGB order, thermal HxW)
# =============================================================================
# Keep these FIXED for the whole project so E1-E4 are scored on identical inputs.
SEVERITY = {
    "low_light": [(0.45, 1.5, 4), (0.30, 2.0, 8), (0.15, 2.5, 12)],  # brightness, gamma, noise std
    "noise":     [10, 20, 35],            # gaussian noise std (0-255)
    "blur":      [2.0, 4.0, 7.0],         # gaussian sigma in px (at 720x1280)
    "occlusion": [0.10, 0.20, 0.35],      # fraction of image area covered by patches
}

CORRUPTIONS = {
    "rgb_low_light":     ("rgb", "low_light"),
    "rgb_noise":         ("rgb", "noise"),
    "rgb_blur":          ("rgb", "blur"),
    "rgb_occlusion":     ("rgb", "occlusion"),      # localized
    "thermal_noise":     ("thermal", "noise"),
    "thermal_blur":      ("thermal", "blur"),
    "thermal_occlusion": ("thermal", "occlusion"),  # localized
}


def _gaussian_noise(img, std, rng):
    return np.clip(img.astype(np.float32) + rng.normal(0.0, std, img.shape), 0, 255).astype(np.uint8)


def _blur(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigmaX=sigma, sigmaY=sigma)


def _low_light(img, brightness, gamma, noise_std, rng):
    x = np.power((img.astype(np.float32) / 255.0) * brightness, gamma) * 255.0
    x = x + rng.normal(0.0, noise_std, img.shape)
    return np.clip(x, 0, 255).astype(np.uint8)


def _occlusion(img, area_frac, rng, fill="noise"):
    out = img.copy()
    h, w = img.shape[:2]
    n = int(rng.integers(3, 7))
    patch_area = area_frac * h * w / n
    for _ in range(n):
        ar = rng.uniform(0.5, 2.0)
        ph = int(min(h, max(4, np.sqrt(patch_area / ar))))
        pw = int(min(w, max(4, patch_area / max(ph, 1))))
        y0, x0 = int(rng.integers(0, h - ph + 1)), int(rng.integers(0, w - pw + 1))
        region = out[y0:y0 + ph, x0:x0 + pw]
        out[y0:y0 + ph, x0:x0 + pw] = rng.integers(0, 256, region.shape) if fill == "noise" else 0
    return out


def apply_corruption(rgb, thermal, name, severity, rng):
    """Corrupt ONE modality. rgb: HxWx3 uint8, thermal: HxW uint8."""
    if name in (None, "clean"):
        return rgb, thermal
    modality, kind = CORRUPTIONS[name]
    s = int(severity) - 1
    x = rgb if modality == "rgb" else thermal
    if kind == "noise":
        x = _gaussian_noise(x, SEVERITY["noise"][s], rng)
    elif kind == "blur":
        x = _blur(x, SEVERITY["blur"][s])
    elif kind == "low_light":
        x = _low_light(x, *SEVERITY["low_light"][s], rng)
    elif kind == "occlusion":
        x = _occlusion(x, SEVERITY["occlusion"][s], rng, "black" if modality == "rgb" else "noise")
    return (x, thermal) if modality == "rgb" else (rgb, x)


def random_training_corruption(rgb, thermal, rng, p_corrupt=0.5):
    """For E3: with prob p_corrupt, corrupt one randomly chosen modality at random severity."""
    if rng.random() > p_corrupt:
        return rgb, thermal, "clean"
    name = str(rng.choice(list(CORRUPTIONS)))
    sev = int(rng.integers(1, 4))
    rgb, thermal = apply_corruption(rgb, thermal, name, sev, rng)
    return rgb, thermal, f"{name}@{sev}"


# =============================================================================
# 2. Dataset
# =============================================================================
def read_rgb(path):
    """Returns RGB-order uint8 HxWx3."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def read_thermal(path):
    """Same as the CMX repo: OpenCV grayscale (16-bit files are reduced to 8-bit)."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def read_label(path):
    lab = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if lab is None:
        raise FileNotFoundError(path)
    if lab.ndim == 3:   # colour-coded label: not an index map -> must be converted first
        raise ValueError(f"{path} is a 3-channel label; convert it to a class-index map first.")
    return lab


def _normalize(img):
    return (img.astype(np.float64) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD


class PST900Dataset(Dataset):
    """
    Layout: <root>/<split>/{rgb,thermal,labels}/*.png
    channel_order: "BGR" to match a model trained with the official CMX repo
                   (its loader feeds BGR), "RGB" for your own pipelines.
    Eval mode returns the FULL-resolution image (sliding-window inference in evaluate()).
    """

    def __init__(self, root_dir, split="test", is_training=False, crop_size=(480, 640),
                 scales=(0.5, 0.75, 1.0, 1.25, 1.5, 1.75), corruption=None, severity=1,
                 seed=0, train_corruption_p=0.0, channel_order="BGR", num_classes=5):
        super().__init__()
        self.is_training, self.crop_size, self.scales = is_training, crop_size, scales
        self.corruption, self.severity, self.seed = corruption, severity, seed
        self.train_corruption_p, self.channel_order = train_corruption_p, channel_order
        self.num_classes = num_classes

        candidates = [os.path.join(root_dir, split),
                      os.path.join(root_dir, "PST900_RGBT_Dataset", split)]
        self.split_dir = next((p for p in candidates if os.path.isdir(p)), None)
        if self.split_dir is None:
            raise FileNotFoundError(f"Cannot locate '{split}' split in {root_dir}")
        self.rgb_dir = os.path.join(self.split_dir, "rgb")
        self.thermal_dir = os.path.join(self.split_dir, "thermal")
        self.label_dir = os.path.join(self.split_dir, "labels")
        self.rgb_files = sorted(glob.glob(os.path.join(self.rgb_dir, "*.png")) +
                                glob.glob(os.path.join(self.rgb_dir, "*.jpg")))
        if not self.rgb_files:
            raise RuntimeError(f"No RGB images in {self.rgb_dir}")

    def __len__(self):
        return len(self.rgb_files)

    @staticmethod
    def find(directory, basename):
        p = os.path.join(directory, basename)
        if os.path.exists(p):
            return p
        c = glob.glob(os.path.join(directory, os.path.splitext(basename)[0] + ".*"))
        if not c:
            raise FileNotFoundError(f"No match for {basename} in {directory}")
        return c[0]

    def load_raw(self, idx):
        f = self.rgb_files[idx]
        name = os.path.basename(f)
        rgb = read_rgb(f)
        th = read_thermal(self.find(self.thermal_dir, name))
        lab = read_label(self.find(self.label_dir, name)).astype(np.int64)
        lab[(lab < 0) | (lab >= self.num_classes)] = IGNORE_INDEX
        return rgb, th, lab, name

    def _train_aug(self, rgb, th, lab):
        if random.random() >= 0.5:
            rgb, th, lab = rgb[:, ::-1], th[:, ::-1], lab[:, ::-1]
        s = random.choice(self.scales)
        h, w = int(rgb.shape[0] * s), int(rgb.shape[1] * s)
        rgb = cv2.resize(np.ascontiguousarray(rgb), (w, h), interpolation=cv2.INTER_LINEAR)
        th = cv2.resize(np.ascontiguousarray(th), (w, h), interpolation=cv2.INTER_LINEAR)
        lab = cv2.resize(np.ascontiguousarray(lab).astype(np.uint8), (w, h),
                         interpolation=cv2.INTER_NEAREST).astype(np.int64)
        ch, cw = self.crop_size
        ph, pw = max(0, ch - h), max(0, cw - w)
        if ph or pw:
            rgb = np.pad(rgb, ((0, ph), (0, pw), (0, 0)))
            th = np.pad(th, ((0, ph), (0, pw)))
            lab = np.pad(lab, ((0, ph), (0, pw)), constant_values=IGNORE_INDEX)  # padded = ignored
        H, W = lab.shape
        y, x = random.randint(0, H - ch), random.randint(0, W - cw)
        return rgb[y:y + ch, x:x + cw], th[y:y + ch, x:x + cw], lab[y:y + ch, x:x + cw]

    def __getitem__(self, idx):
        rgb, th, lab, name = self.load_raw(idx)

        if self.is_training and self.train_corruption_p > 0:
            rgb, th, _ = random_training_corruption(rgb, th, np.random.default_rng(),
                                                    self.train_corruption_p)
        elif not self.is_training and self.corruption not in (None, "clean"):
            rng = np.random.default_rng(self.seed * 100003 + idx)  # identical every run
            rgb, th = apply_corruption(rgb, th, self.corruption, self.severity, rng)

        if self.is_training:
            rgb, th, lab = self._train_aug(rgb, th, lab)

        if self.channel_order == "BGR":
            rgb = rgb[..., ::-1]
        th3 = np.stack([th] * 3, axis=-1)
        rgb_t = torch.from_numpy(np.ascontiguousarray(_normalize(rgb).transpose(2, 0, 1))).float()
        th_t = torch.from_numpy(np.ascontiguousarray(_normalize(th3).transpose(2, 0, 1))).float()
        return {"rgb": rgb_t, "thermal": th_t,
                "label": torch.from_numpy(np.ascontiguousarray(lab)).long(), "name": name}


def inspect_dataset(root_dir, split="train", n=50):
    """Prints label values, image modes and sizes. Run once before training."""
    ds = PST900Dataset(root_dir, split)
    vals, info = set(), {}
    for f in ds.rgb_files[:n]:
        name = os.path.basename(f)
        lab = cv2.imread(ds.find(ds.label_dir, name), cv2.IMREAD_UNCHANGED)
        th = cv2.imread(ds.find(ds.thermal_dir, name), cv2.IMREAD_UNCHANGED)
        rgb = cv2.imread(f, cv2.IMREAD_UNCHANGED)
        vals |= set(np.unique(lab).tolist())
        info = {"rgb": (rgb.shape, rgb.dtype), "thermal": (th.shape, th.dtype, int(th.min()), int(th.max())),
                "label": (lab.shape, lab.dtype)}
    print(f"[{split}] {len(ds)} samples")
    print(f"  label values (first {n}): {sorted(vals)}   <- expect a subset of 0..4 (and maybe 255)")
    for k, v in info.items():
        print(f"  {k:8s}: {v}")
    if any(v >= ds.num_classes and v != IGNORE_INDEX for v in vals):
        print("  WARNING: label values outside 0..4 -> check the label encoding before training!")
    return sorted(vals)


# =============================================================================
# 3. Metrics
# =============================================================================
class SegMetrics:
    """Confusion matrix: rows = ground truth, cols = prediction."""

    def __init__(self, num_classes, class_names=None, ignore_index=IGNORE_INDEX, bg_index=0):
        self.n = num_classes
        self.names = class_names or [str(i) for i in range(num_classes)]
        self.ignore_index, self.bg_index = ignore_index, bg_index
        self.cm = torch.zeros(num_classes, num_classes, dtype=torch.float64)

    @torch.no_grad()
    def update(self, pred, target):
        pred, target = pred.reshape(-1).cpu(), target.reshape(-1).cpu()
        keep = target != self.ignore_index
        idx = target[keep] * self.n + pred[keep]
        self.cm += torch.bincount(idx, minlength=self.n ** 2).view(self.n, self.n).double()

    def compute(self):
        cm = self.cm
        tp = cm.diag()
        fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
        tn = cm.sum() - tp - fp - fn
        present = cm.sum(1) > 0

        def safe(a, b):
            return torch.where(b > 0, a / b.clamp(min=1e-12), torch.full_like(a, float("nan")))

        per = {"IoU": safe(tp, tp + fp + fn), "Dice": safe(2 * tp, 2 * tp + fp + fn),
               "Sensitivity": safe(tp, tp + fn), "Specificity": safe(tn, tn + fp)}
        obj = torch.tensor([i != self.bg_index for i in range(self.n)]) & present
        res = {"pixel_acc": (tp.sum() / cm.sum()).item()}
        for k, v in per.items():
            short = "mIoU" if k == "IoU" else f"m{k}"
            res[short] = torch.nanmean(v[present]).item()
            res[f"{short}_obj"] = torch.nanmean(v[obj]).item()
            for i, nm in enumerate(self.names):
                res[f"{k}/{nm}"] = v[i].item()
        return res


# =============================================================================
# 4. Inference & evaluation
# =============================================================================
def _default_forward(model, rgb, thermal):
    out = model(rgb, thermal)
    return out[0] if isinstance(out, (tuple, list)) else out


@torch.no_grad()
def predict_probs(model, rgb, th, forward_fn=_default_forward, crop=(480, 640),
                  stride_rate=2 / 3, flip=False):
    """Sliding-window softmax probabilities at input resolution (B x C x H x W)."""
    def run(r, t):
        p = forward_fn(model, r, t).float().softmax(1)
        if flip:
            p = p + forward_fn(model, r.flip(-1), t.flip(-1)).float().softmax(1).flip(-1)
        return p

    B, _, H, W = rgb.shape
    if crop is None:
        return run(rgb, th)
    ch, cw = crop
    ph, pw = max(0, ch - H), max(0, cw - W)
    if ph or pw:
        rgb, th = F.pad(rgb, (0, pw, 0, ph)), F.pad(th, (0, pw, 0, ph))
    Hp, Wp = rgb.shape[-2:]
    sh, sw = int(np.ceil(ch * stride_rate)), int(np.ceil(cw * stride_rate))
    ys = list(range(0, max(Hp - ch, 0) + 1, sh))
    xs = list(range(0, max(Wp - cw, 0) + 1, sw))
    if ys[-1] != Hp - ch:
        ys.append(Hp - ch)
    if xs[-1] != Wp - cw:
        xs.append(Wp - cw)
    probs, count = None, torch.zeros(1, 1, Hp, Wp, device=rgb.device)
    for y in ys:
        for x in xs:
            p = run(rgb[..., y:y + ch, x:x + cw], th[..., y:y + ch, x:x + cw])
            if probs is None:
                probs = torch.zeros(B, p.shape[1], Hp, Wp, device=rgb.device)
            probs[..., y:y + ch, x:x + cw] += p
            count[..., y:y + ch, x:x + cw] += 1
    return (probs / count)[..., :H, :W]


@torch.no_grad()
def evaluate(model, loader, num_classes=5, forward_fn=_default_forward, device=None,
             crop=(480, 640), stride_rate=2 / 3, flip=False, class_names=PST900_CLASSES):
    device = device or next(model.parameters()).device
    model.eval()
    metrics = SegMetrics(num_classes, class_names)
    for batch in loader:
        rgb, th = batch["rgb"].to(device), batch["thermal"].to(device)
        label = batch["label"].to(device)
        probs = predict_probs(model, rgb, th, forward_fn, crop, stride_rate, flip)
        if probs.shape[-2:] != label.shape[-2:]:
            probs = F.interpolate(probs, size=label.shape[-2:], mode="bilinear", align_corners=False)
        metrics.update(probs.argmax(1), label)
    return metrics.compute()


def run_benchmark(model, root_dir, forward_fn=_default_forward, corruptions=None,
                  severities=(1, 2, 3), crop=(480, 640), stride_rate=2 / 3, flip=False,
                  channel_order="BGR", num_workers=2, seed=0, out_csv=None, tag="", verbose=True):
    """Clean + every corruption x severity on the test split. Returns list of dicts."""
    corruptions = corruptions if corruptions is not None else list(CORRUPTIONS)
    conditions = [("clean", 0)] + [(c, s) for c in corruptions for s in severities]
    rows = []
    for cname, sev in conditions:
        ds = PST900Dataset(root_dir, "test", corruption=cname, severity=max(sev, 1),
                           seed=seed, channel_order=channel_order)
        dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=num_workers)
        r = evaluate(model, dl, len(PST900_CLASSES), forward_fn, crop=crop,
                     stride_rate=stride_rate, flip=flip)
        rows.append({"tag": tag, "condition": cname, "severity": sev, **r})
        if verbose:
            print(f"{tag:>6s} {cname:>18s} s{sev} | mIoU {r['mIoU']*100:6.2f} | mDice {r['mDice']*100:6.2f}"
                  f" | mSens {r['mSensitivity']*100:6.2f} | mSpec {r['mSpecificity']*100:6.2f}"
                  f" | mIoU(obj) {r['mIoU_obj']*100:6.2f}")
    if out_csv:
        new = not os.path.exists(out_csv)
        with open(out_csv, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            if new:
                w.writeheader()
            w.writerows(rows)
        if verbose:
            print(f"appended {len(rows)} rows to {out_csv}")
    return rows


def robustness_summary(rows):
    clean = next(r for r in rows if r["condition"] == "clean")
    corr = [r for r in rows if r["condition"] != "clean"]
    out = {}
    for k in ("mIoU", "mDice", "mSensitivity", "mSpecificity", "mIoU_obj"):
        avg = float(np.mean([r[k] for r in corr]))
        out[f"{k}_clean"], out[f"{k}_corrupt_avg"], out[f"{k}_drop"] = clean[k], avg, clean[k] - avg
    for mod in ("rgb", "thermal"):
        sub = [r["mIoU"] for r in corr if r["condition"].startswith(mod)]
        out[f"mIoU_{mod}_degraded_avg"] = float(np.mean(sub)) if sub else float("nan")
    return out


# =============================================================================
# 5. Cost
# =============================================================================
@torch.no_grad()
def model_cost(model, input_size=(480, 640), forward_fn=_default_forward, device=None,
               warmup=10, iters=50):
    device = device or next(model.parameters()).device
    model.eval()
    rgb = torch.randn(1, 3, *input_size, device=device)
    th = torch.randn(1, 3, *input_size, device=device)
    res = {"params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    try:
        from fvcore.nn import FlopCountAnalysis

        class _Wrap(torch.nn.Module):
            def __init__(s, m):
                super().__init__()
                s.m = m

            def forward(s, a, b):
                return forward_fn(s.m, a, b)

        fl = FlopCountAnalysis(_Wrap(model), (rgb, th))
        fl.unsupported_ops_warnings(False)
        fl.uncalled_modules_warnings(False)
        res["GFLOPs"] = fl.total() / 1e9
    except Exception as e:
        res["GFLOPs"] = f"n/a ({type(e).__name__})"
    for _ in range(warmup):
        forward_fn(model, rgb, th)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        forward_fn(model, rgb, th)
    if device.type == "cuda":
        torch.cuda.synchronize()
    res["latency_ms"] = (time.perf_counter() - t0) / iters * 1000
    res["FPS"] = 1000 / res["latency_ms"]
    return res


# =============================================================================
# 6. Visual helpers
# =============================================================================
def colorize(label):
    out = np.zeros((*label.shape, 3), np.uint8)
    for i, c in enumerate(PST900_PALETTE):
        out[label == i] = c
    out[label == IGNORE_INDEX] = [128, 128, 128]
    return out


def corruption_grid(root_dir, idx=0, severity=2):
    """Returns an image: rows = clean + each corruption; columns = RGB | thermal."""
    ds = PST900Dataset(root_dir, "test")
    rgb0, th0, _, _ = ds.load_raw(idx)
    rows = []
    for c in ["clean"] + list(CORRUPTIONS):
        r, t = apply_corruption(rgb0, th0, c, severity, np.random.default_rng(idx))
        rows.append(np.concatenate([r, np.stack([t] * 3, -1)], axis=1))
    return np.concatenate(rows, axis=0), ["clean"] + list(CORRUPTIONS)

In [ ]:
%%writefile /content/cmx_colab.py
"""
cmx_colab.py -- helpers to run the OFFICIAL CMX repo (huaaaliu/RGBX_Semantic_Segmentation)
on PST900 in Google Colab (single GPU, resumable from Google Drive).
"""

import os
import re
import sys
import glob
import shutil

PST900_CLASSES = ["Background", "Fire-Extinguisher", "Backpack", "Hand-Drill", "Survivor"]


# -----------------------------------------------------------------------------
# 1. Patch the repo so it runs on modern PyTorch / single-GPU Colab.
#    These are bug fixes only; the model, loss, optimizer and LR schedule are untouched.
# -----------------------------------------------------------------------------
def patch_repo(repo_dir):
    def edit(rel, pairs):
        p = os.path.join(repo_dir, rel)
        src = open(p).read()
        for old, new in pairs:
            if new in src:
                continue                      # already patched
            if old not in src:
                raise RuntimeError(f"Patch target not found in {rel}: {old!r}")
            src = src.replace(old, new)
        open(p, "w").write(src)

    edit("train.py", [
        # Python-3 iterators have no .next()
        ("minibatch = dataloader.next()", "minibatch = next(dataloader)"),
        # `sum_loss += loss` keeps every iteration's autograd graph alive -> GPU memory leak
        ("                sum_loss += loss\n",
         "                sum_loss += loss.item()\n"),
        ("' loss=%.4f total_loss=%.4f' % (loss, (sum_loss / (idx + 1)))",
         "' loss=%.4f total_loss=%.4f' % (loss.item(), (sum_loss / (idx + 1)))"),
    ])
    edit("engine/engine.py", [
        # resuming a NON-distributed run must not prefix keys with 'module.'
        ("load_model(self.state.model, tmp['model'], is_restore=True)",
         "load_model(self.state.model, tmp['model'], is_restore=self.distributed)"),
        ("tmp = torch.load(self.continue_state_object)\n",
         "tmp = torch.load(self.continue_state_object, map_location='cpu', weights_only=False)\n"),
    ])
    edit("engine/engine.py", [
        # each checkpoint is ~800 MB (model + AdamW state): keep only the newest 2 on Drive
        ("        self.save_checkpoint(current_epoch_checkpoint)\n",
         "        self.save_checkpoint(current_epoch_checkpoint)\n"
         "        import glob as _g, re as _re\n"
         "        _old = sorted(_g.glob(osp.join(checkpoint_dir, 'epoch-*.pth')),\n"
         "                      key=lambda p: int(_re.search(r'epoch-(\\d+)', p).group(1))\n"
         "                      if _re.search(r'epoch-(\\d+)\\.pth$', p) else -1)\n"
         "        for _p in [p for p in _old if _re.search(r'epoch-(\\d+)\\.pth$', p)][:-2]:\n"
         "            os.remove(_p)\n"),
    ])
    edit("utils/transforms.py", [
        # removed in Python 3.10
        ("isinstance(shape, collections.Iterable)", "isinstance(shape, collections.abc.Iterable)"),
        ("import collections\n", "import collections\nimport collections.abc\n"),
    ])
    print("Repo patched: next(), loss-graph leak, single-GPU resume, torch.load, collections.abc.")


# -----------------------------------------------------------------------------
# 2. Convert PST900 to the CMX folder layout (symlinks, no copying).
#    <dst>/RGB, <dst>/Thermal, <dst>/Label, train.txt, test.txt
#    Names are prefixed with the split so train/test files can never collide.
# -----------------------------------------------------------------------------
def prepare_pst900_for_cmx(src_root, dst):
    split_root = src_root
    if not os.path.isdir(os.path.join(split_root, "train")):
        split_root = os.path.join(src_root, "PST900_RGBT_Dataset")
    for sub in ("RGB", "Thermal", "Label"):
        os.makedirs(os.path.join(dst, sub), exist_ok=True)
    counts = {}
    for split in ("train", "test"):
        names = []
        rgb_files = sorted(glob.glob(os.path.join(split_root, split, "rgb", "*.png")))
        for f in rgb_files:
            stem = os.path.splitext(os.path.basename(f))[0]
            new = f"{split}_{stem}"
            pairs = {"RGB": f,
                     "Thermal": os.path.join(split_root, split, "thermal", stem + ".png"),
                     "Label": os.path.join(split_root, split, "labels", stem + ".png")}
            for sub, srcf in pairs.items():
                if not os.path.exists(srcf):
                    raise FileNotFoundError(srcf)
                link = os.path.join(dst, sub, new + ".png")
                if not os.path.lexists(link):
                    os.symlink(os.path.abspath(srcf), link)
            names.append(new)
        with open(os.path.join(dst, f"{split}.txt"), "w") as fh:
            fh.write("\n".join(names) + "\n")
        counts[split] = len(names)
    print(f"CMX-format dataset at {dst}: {counts}")
    return counts


# -----------------------------------------------------------------------------
# 3. Write config.py for PST900 (same structure/defaults as the repo's config.py).
# -----------------------------------------------------------------------------
CONFIG_TEMPLATE = '''import os
import os.path as osp
import sys
import time
import numpy as np
from easydict import EasyDict as edict

C = edict()
config = C
cfg = C

C.seed = {seed}

C.root_dir = os.path.abspath(os.path.join(os.getcwd(), './'))
C.abs_dir = osp.realpath(".")

"""Dataset Path"""
C.dataset_name = 'PST900'
C.dataset_path = r'{dataset_path}'
C.rgb_root_folder = osp.join(C.dataset_path, 'RGB')
C.rgb_format = '.png'
C.gt_root_folder = osp.join(C.dataset_path, 'Label')
C.gt_format = '.png'
C.gt_transform = False          # PST900: label 0 = Background is a VALID class (do not shift)
C.x_root_folder = osp.join(C.dataset_path, 'Thermal')
C.x_format = '.png'
C.x_is_single_channel = True    # thermal is single channel -> replicated to 3 channels
C.train_source = osp.join(C.dataset_path, "train.txt")
C.eval_source = osp.join(C.dataset_path, "test.txt")
C.is_test = False
C.num_train_imgs = {num_train}
C.num_eval_imgs = {num_eval}
C.num_classes = 5
C.class_names = {class_names}

"""Image Config"""
C.background = 255
C.image_height = {crop_h}
C.image_width = {crop_w}
C.norm_mean = np.array([0.485, 0.456, 0.406])
C.norm_std = np.array([0.229, 0.224, 0.225])

"""Network"""
C.backbone = '{backbone}'
C.pretrained_model = r'{pretrained}'
C.decoder = 'MLPDecoder'
C.decoder_embed_dim = 512
C.optimizer = 'AdamW'

"""Train Config"""
C.lr = {lr}
C.lr_power = 0.9
C.momentum = 0.9
C.weight_decay = 0.01
C.batch_size = {batch_size}
C.nepochs = {nepochs}
C.niters_per_epoch = C.num_train_imgs // C.batch_size + 1
C.num_workers = {num_workers}
C.train_scale_array = [0.5, 0.75, 1, 1.25, 1.5, 1.75]
C.warm_up_epoch = 10

C.fix_bias = True
C.bn_eps = 1e-3
C.bn_momentum = 0.1

"""Eval Config"""
C.eval_iter = 25
C.eval_stride_rate = 2 / 3
C.eval_scale_array = [1]
C.eval_flip = False
C.eval_crop_size = [{crop_h}, {crop_w}]

"""Store Config"""
C.checkpoint_start_epoch = {ckpt_every}
C.checkpoint_step = {ckpt_every}

"""Path Config"""
def add_path(path):
    if path not in sys.path:
        sys.path.insert(0, path)
add_path(osp.join(C.root_dir))

C.log_dir = r'{log_dir}'
C.tb_dir = osp.abspath(osp.join(C.log_dir, "tb"))
C.log_dir_link = C.log_dir
C.checkpoint_dir = osp.abspath(osp.join(C.log_dir, "checkpoint"))

exp_time = time.strftime('%Y_%m_%d_%H_%M_%S', time.localtime())
C.log_file = C.log_dir + '/log_' + exp_time + '.log'
C.link_log_file = C.log_file + '/log_last.log'
C.val_log_file = C.log_dir + '/val_' + exp_time + '.log'
C.link_val_log_file = C.log_dir + '/val_last.log'
'''


def write_config(repo_dir, dataset_path, pretrained, log_dir, num_train, num_eval,
                 backbone="mit_b2", batch_size=8, nepochs=500, lr=6e-5, crop=(480, 640),
                 num_workers=2, ckpt_every=10, seed=12345):
    os.makedirs(log_dir, exist_ok=True)
    txt = CONFIG_TEMPLATE.format(
        seed=seed, dataset_path=dataset_path, num_train=num_train, num_eval=num_eval,
        class_names=PST900_CLASSES, crop_h=crop[0], crop_w=crop[1], backbone=backbone,
        pretrained=pretrained, lr=lr, batch_size=batch_size, nepochs=nepochs,
        num_workers=num_workers, ckpt_every=ckpt_every, log_dir=log_dir)
    path = os.path.join(repo_dir, "config.py")
    if os.path.exists(path) and not os.path.exists(path + ".orig"):
        shutil.copy(path, path + ".orig")
    open(path, "w").write(txt)
    print(f"Wrote {path}  (bs={batch_size}, epochs={nepochs}, lr={lr}, ckpt every {ckpt_every})")


# -----------------------------------------------------------------------------
# 4. Model helpers
# -----------------------------------------------------------------------------
def _import_repo(repo_dir):
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    for m in [k for k in list(sys.modules) if k == "config" or k.startswith(("models", "engine", "utils"))]:
        del sys.modules[m]                       # force re-import of the freshly written config
    cwd = os.getcwd()
    os.chdir(repo_dir)
    try:
        from config import config
        from models.builder import EncoderDecoder
    finally:
        os.chdir(cwd)
    return config, EncoderDecoder


def build_model(repo_dir, checkpoint=None, device="cuda", with_pretrained=False):
    """CMX EncoderDecoder from the repo; loads a training checkpoint if given."""
    import torch
    import torch.nn as nn
    config, EncoderDecoder = _import_repo(repo_dir)
    crit = nn.CrossEntropyLoss(ignore_index=255) if with_pretrained else None
    model = EncoderDecoder(cfg=config, criterion=crit, norm_layer=nn.BatchNorm2d)
    if checkpoint:
        sd = torch.load(checkpoint, map_location="cpu", weights_only=False)
        sd = sd.get("model", sd)
        sd = {k[7:] if k.startswith("module.") else k: v for k, v in sd.items()}
        model.load_state_dict(sd, strict=True)
        print(f"Loaded checkpoint {os.path.basename(checkpoint)}")
    return model.to(device).eval()


def verify_pretrained(repo_dir, pretrained_path):
    """The repo loads ImageNet weights with strict=False, which hides failures. Check it."""
    import torch
    raw = torch.load(pretrained_path, map_location="cpu", weights_only=False)
    raw = raw.get("model", raw)
    model = build_model(repo_dir, None, device="cpu", with_pretrained=True)
    own = model.backbone.state_dict()
    hit = [k for k in raw if k in own and own[k].shape == raw[k].shape
           and torch.equal(own[k].cpu(), raw[k].cpu())]
    enc_keys = [k for k in raw if any(s in k for s in ("patch_embed", "block", "norm"))]
    print(f"Pretrained keys loaded into RGB stream: {len(hit)}/{len(enc_keys)}")
    if len(hit) < 0.9 * len(enc_keys):
        raise RuntimeError("Pretrained weights did NOT load -- check the file/backbone name.")
    return len(hit), len(enc_keys)


def latest_checkpoint(log_dir):
    ck = glob.glob(os.path.join(log_dir, "checkpoint", "epoch-*.pth"))
    ck = [c for c in ck if re.search(r"epoch-(\d+)\.pth$", c)]
    if not ck:
        return None
    return max(ck, key=lambda c: int(re.search(r"epoch-(\d+)\.pth$", c).group(1)))

In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import rgbt_eval, cmx_colab
importlib.reload(rgbt_eval); importlib.reload(cmx_colab)
from rgbt_eval import *
from cmx_colab import *
print('toolkit loaded')

## 6. Inspect the data (do not skip)
Checks that:
* labels are class indices **0–4** (255 = ignore is fine);
* thermal bit depth (16-bit thermal is reduced to 8-bit by OpenCV, same as the CMX repo);
* RGB, thermal and label are aligned.

**Also confirm the class order against the official PST900 README.** Every per-class number depends on it.

In [ ]:
inspect_dataset(PST_ROOT, 'train')
inspect_dataset(PST_ROOT, 'test')

In [ ]:
import matplotlib.pyplot as plt
ds_vis = PST900Dataset(PST_ROOT, 'train')
fig, ax = plt.subplots(3, 4, figsize=(18, 9))
for r, i in enumerate([0, len(ds_vis)//2, len(ds_vis)-1]):
    rgb, th, lab, name = ds_vis.load_raw(i)
    ov = (0.55*rgb + 0.45*colorize(lab)).astype('uint8')
    for c, (img, t) in enumerate([(rgb, f'RGB {name}'), (th, 'Thermal'), (colorize(lab), 'Label'), (ov, 'Overlay')]):
        ax[r, c].imshow(img, cmap='inferno' if img.ndim == 2 else None); ax[r, c].set_title(t); ax[r, c].axis('off')
plt.suptitle('Black=Background  Red=Fire-Ext  Green=Backpack  Blue=Hand-Drill  Yellow=Survivor')
plt.tight_layout(); plt.show()

## 7. Corruption benchmark preview
Look at severity 1 and 3. If a corruption looks unrealistic, tune `SEVERITY` in `rgbt_eval.py` **now**.
After you report E1 numbers, keep the benchmark fixed for E1–E4.

In [ ]:
for sev in (1, 3):
    grid, names = corruption_grid(PST_ROOT, idx=5, severity=sev)
    h = grid.shape[0] // len(names)
    fig, ax = plt.subplots(len(names), 1, figsize=(10, 2.6*len(names)))
    for k, n in enumerate(names):
        ax[k].imshow(grid[k*h:(k+1)*h]); ax[k].set_title(f'{n}  (severity {sev})  |  left: RGB, right: thermal'); ax[k].axis('off')
    plt.tight_layout(); plt.show()

## 8. Official CMX repo + Colab fixes
The fixes are bug fixes only. The model, loss, optimizer and LR schedule are **unchanged**:
1. `dataloader.next()` → `next(dataloader)` (Python 3)
2. `sum_loss += loss` → `loss.item()` (it kept every iteration's graph in GPU memory)
3. single-GPU resume (`is_restore=self.distributed`) + `torch.load(weights_only=False)`
4. `collections.Iterable` → `collections.abc.Iterable` (removed in Python 3.10)
5. keep only the newest 2 checkpoints (each is ~800 MB on Drive)

In [ ]:
if not os.path.exists(REPO):
    !git clone -q https://github.com/huaaaliu/RGBX_Semantic_Segmentation.git {REPO}
patch_repo(REPO)

## 8b. Add CMRF to CMX (the E2 change)
* `cmrf_src.py` contains the module and a patch that inserts it into the official code:
  * `models/encoders/dual_segformer.py`: CMRF runs just before `self.FRMs[k]` at the configured stage.
  * `models/builder.py`: adds the auxiliary prediction loss to the training loss.
* The stage and the hyper-parameters are stored in `cmrf_config.json` inside the repo. **With an empty stage list the model is exactly E1.**

In [ ]:
%%writefile /content/cmrf_src.py
"""
CMRF: Cross-Modal Residual Filtering (E2 hypothesis module) for CMX.

At one encoder stage, before CMX's CM-FRM:
  1. Predict each modality's features from the other   T_hat = P_r2t(F_rgb),  R_hat = P_t2r(F_thm)
  2. Residual = what the other modality cannot explain  Res_t = F_thm - T_hat,  Res_r = F_rgb - R_hat
  3. Gate (per pixel, per channel), from |residual| and the receiving stream
  4. Inject the gated residual into the OTHER stream     F_rgb' = F_rgb + s_t * G_t * proj(Res_t)
                                                         F_thm' = F_thm + s_r * G_r * proj(Res_r)
Auxiliary loss (training only): normalised MSE between predictions and stop-gradient targets,
so the predictors learn real cross-modal correspondence and features cannot collapse.
"""
import json
import os.path as osp

import torch
import torch.nn as nn


def load_cmrf_config():
    """Reads <repo>/cmrf_config.json. Missing file -> CMRF disabled (plain CMX = E1)."""
    p = osp.join(osp.dirname(osp.abspath(__file__)), '..', 'cmrf_config.json')
    if not osp.exists(p):
        return {'stages': []}
    with open(p) as f:
        return json.load(f)


class _Predictor(nn.Sequential):
    """Lightweight cross-modal predictor: 1x1 -> depthwise 3x3 -> 1x1."""
    def __init__(self, dim, hidden):
        super().__init__(
            nn.Conv2d(dim, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False), nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Conv2d(hidden, dim, 1))


class _Gate(nn.Sequential):
    """Per-pixel, per-channel gate in [0, 1] from [|residual|, receiving features]."""
    def __init__(self, dim, hidden):
        super().__init__(
            nn.Conv2d(2 * dim, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Conv2d(hidden, dim, 1), nn.Sigmoid())


class CMRF(nn.Module):
    def __init__(self, dim, reduction=4, lam=0.1, init_scale=0.1):
        super().__init__()
        hidden = max(dim // reduction, 16)
        self.pred_r2t = _Predictor(dim, hidden)   # RGB -> thermal
        self.pred_t2r = _Predictor(dim, hidden)   # thermal -> RGB
        self.gate_t = _Gate(dim, hidden)          # how much thermal-unique info enters the RGB stream
        self.gate_r = _Gate(dim, hidden)          # how much RGB-unique info enters the thermal stream
        self.proj_t = nn.Conv2d(dim, dim, 1)
        self.proj_r = nn.Conv2d(dim, dim, 1)
        self.scale_t = nn.Parameter(torch.full((1,), float(init_scale)))
        self.scale_r = nn.Parameter(torch.full((1,), float(init_scale)))
        self.lam = lam
        self.aux_loss = None
        self.last = {}                            # filled in eval mode, for visualisation

    @staticmethod
    def _nmse(pred, target):
        target = target.detach()                  # stop-gradient on the target
        return ((pred - target) ** 2).mean() / (target.pow(2).mean() + 1e-6)

    def forward(self, x_rgb, x_thm):
        t_hat = self.pred_r2t(x_rgb)
        r_hat = self.pred_t2r(x_thm)
        res_t = x_thm - t_hat                     # thermal information RGB cannot explain
        res_r = x_rgb - r_hat                     # RGB information thermal cannot explain
        g_t = self.gate_t(torch.cat([res_t.abs(), x_rgb], 1))
        g_r = self.gate_r(torch.cat([res_r.abs(), x_thm], 1))
        out_rgb = x_rgb + self.scale_t * g_t * self.proj_t(res_t)
        out_thm = x_thm + self.scale_r * g_r * self.proj_r(res_r)

        if self.training:
            self.aux_loss = self.lam * (self._nmse(t_hat, x_thm) + self._nmse(r_hat, x_rgb))
            self.last = {}
        else:
            self.aux_loss = None
            self.last = {'gate_t': g_t.mean(1).detach(), 'gate_r': g_r.mean(1).detach(),
                         'res_t': res_t.abs().mean(1).detach(), 'res_r': res_r.abs().mean(1).detach()}
        return out_rgb, out_thm


def patch_cmx_for_cmrf(repo_dir):
    """Adds CMRF to the official CMX code. Idempotent. Stages come from cmrf_config.json."""
    import shutil
    shutil.copy(osp.abspath(__file__), osp.join(repo_dir, 'models', 'cmrf.py'))

    p = osp.join(repo_dir, 'models', 'encoders', 'dual_segformer.py')
    s = open(p).read()
    if '[CMRF]' not in s:
        s = s.replace('import math\n', 'import math\nfrom ..cmrf import CMRF, load_cmrf_config  # [CMRF]\n', 1)
        anchor = ('norm_layer=norm_fuse)])\n\n'
                  '        self.apply(self._init_weights)\n')    # end of the FFM list in RGBXTransformer
        assert s.count(anchor) == 1, 'unexpected dual_segformer.py layout'
        s = s.replace(anchor, ('norm_layer=norm_fuse)])\n\n' +
            "        # [CMRF] cross-modal residual filtering before CM-FRM at the configured stages\n"
            "        _cc = load_cmrf_config()\n"
            "        self.cmrf_stages = [int(k) for k in _cc.get('stages', [])]\n"
            "        self.CMRFs = nn.ModuleDict({str(k): CMRF(embed_dims[k - 1], reduction=_cc.get('reduction', 4),\n"
            "                                                lam=_cc.get('lambda', 0.1), init_scale=_cc.get('init_scale', 0.1))\n"
            "                                    for k in self.cmrf_stages})\n"
            "        self.apply(self._init_weights)\n"), 1)
        for k in range(4):
            call = f'        x_rgb, x_e = self.FRMs[{k}](x_rgb, x_e)\n'
            assert s.count(call) == 1, f'FRM call for stage {k + 1} not found'
            s = s.replace(call, (f"        if '{k + 1}' in self.CMRFs:  # [CMRF]\n"
                                 f"            x_rgb, x_e = self.CMRFs['{k + 1}'](x_rgb, x_e)\n" + call), 1)
        s = s.replace('    def init_weights(self, pretrained=None):\n', (
            "    def cmrf_aux_loss(self):  # [CMRF]\n"
            "        ls = [m.aux_loss for m in self.CMRFs.values() if m.aux_loss is not None]\n"
            "        return sum(ls) if ls else None\n\n"
            "    def init_weights(self, pretrained=None):\n"), 1)
        open(p, 'w').write(s)

    p = osp.join(repo_dir, 'models', 'builder.py')
    s = open(p).read()
    if '[CMRF]' not in s:
        old = '            loss = self.criterion(out, label.long())\n'
        assert s.count(old) == 1, 'unexpected builder.py layout'
        s = s.replace(old, old + (
            "            _aux = self.backbone.cmrf_aux_loss() if hasattr(self.backbone, 'cmrf_aux_loss') else None  # [CMRF]\n"
            "            if _aux is not None:\n"
            "                loss = loss + _aux\n"), 1)
        open(p, 'w').write(s)
    print('CMX patched with CMRF (stages read from cmrf_config.json)')


def write_cmrf_config(repo_dir, stages=(3,), reduction=4, lam=0.1, init_scale=0.1):
    cfg = {'stages': list(stages), 'reduction': reduction, 'lambda': lam, 'init_scale': init_scale}
    with open(osp.join(repo_dir, 'cmrf_config.json'), 'w') as f:
        json.dump(cfg, f, indent=2)
    print('cmrf_config.json:', cfg)
    return cfg


In [ ]:
# ---- E2 settings ----
CMRF_STAGES     = [3]     # encoder stage(s) that get CMRF (1-4); start with a single stage, as planned
CMRF_REDUCTION  = 4       # hidden width = channels / 4 (keeps it lightweight)
CMRF_LAMBDA     = 0.1     # weight of the cross-modal prediction loss
CMRF_INIT_SCALE = 0.1     # initial strength of the injected residual

import sys, importlib
sys.path.insert(0, '/content')
import cmrf_src; importlib.reload(cmrf_src)
cmrf_src.patch_cmx_for_cmrf(REPO)
CMRF_CFG = cmrf_src.write_cmrf_config(REPO, stages=CMRF_STAGES, reduction=CMRF_REDUCTION,
                                      lam=CMRF_LAMBDA, init_scale=CMRF_INIT_SCALE)

## 9. PST900 → CMX folder layout
Builds `RGB/ Thermal/ Label/ train.txt test.txt` using symlinks (no copying).

Two config settings matter for PST900:
* `gt_transform=False`: the repo's default subtracts 1 from every label, which would turn Background into 255.
* `x_is_single_channel=True`: thermal is replicated to 3 channels.

In [ ]:
counts = prepare_pst900_for_cmx(PST_ROOT, CMX_DATA)
assert counts == {'train': 597, 'test': 288}, f'Unexpected split sizes {counts} (PST900 should be 597/288)'

## 10. ImageNet-pretrained MiT-B2 (SegFormer) weights
Downloaded once from the repo's Google-Drive folder into your Drive.
If `gdown` fails (Drive quota), open the folder link from the repo README, download `mit_b2.pth` manually,
and put it in `MyDrive/AIProj_CMX/pretrained/`.

In [ ]:
import glob, shutil
MIT_B2 = f'{PRETRAIN}/mit_b2.pth'
if not os.path.exists(MIT_B2):
    !gdown --folder https://drive.google.com/drive/folders/10XgSW8f7ghRs9fJ0dE-EV8G2E_guVsT5 -O /content/pretrained_dl
    found = glob.glob('/content/pretrained_dl/**/mit_b2.pth', recursive=True)
    assert found, 'mit_b2.pth not found: download it manually (see text above).'
    shutil.copy(found[0], MIT_B2)
print('MiT-B2 weights:', MIT_B2, f'{os.path.getsize(MIT_B2)/1e6:.0f} MB')

## 11. Training configuration
These follow the repo defaults (AdamW, lr 6e-5, poly decay, 10 warm-up epochs, scales 0.5–1.75, 480×640 crops).

* **`BATCH_SIZE`:** the repo default is 8. On a 16 GB T4, use 4 if the smoke test runs out of memory. **Record any change.**
* **`NEPOCHS`:** check the CMX paper's RGB-T implementation details and match them. 500 is the repo default.

In [ ]:
BATCH_SIZE = 4        # 8 = repo default; drop to 4 if the smoke test OOMs
NEPOCHS    = 100      # agreed training schedule
LR         = 6e-5
CROP       = (480, 640)
CKPT_EVERY = 5        # epochs between checkpoints (only newest 2 are kept)

write_config(REPO, CMX_DATA, MIT_B2, LOG_DIR, counts['train'], counts['test'],
             backbone='mit_b2', batch_size=BATCH_SIZE, nepochs=NEPOCHS, lr=LR,
             crop=CROP, num_workers=2, ckpt_every=CKPT_EVERY)

# The repo loads ImageNet weights with strict=False, which hides failures, so verify it explicitly.
verify_pretrained(REPO, MIT_B2)

### 11b. Check the CMRF model before training
Builds E2, confirms CMRF is present at the configured stage, and measures the extra parameters and GFLOPs against E1 (66.56 M, 78.27 GFLOPs).

In [ ]:
import torch
_m = build_model(REPO, None, device='cpu', with_pretrained=True)
assert len(_m.backbone.CMRFs) == len(CMRF_STAGES), 'CMRF not found in the model: re-run section 8b'
p_total = sum(p.numel() for p in _m.parameters()) / 1e6
p_cmrf = sum(p.numel() for p in _m.backbone.CMRFs.parameters()) / 1e6
print(f'CMRF at stage(s) {CMRF_STAGES} | total {p_total:.2f} M params | CMRF {p_cmrf:.3f} M (+{100*p_cmrf/66.56:.2f}% vs E1)')
with torch.no_grad():
    _o = _m.eval()(torch.randn(1, 3, *CROP), torch.randn(1, 3, *CROP))
print('forward OK, output', tuple(_o.shape))
del _m, _o

## 12. Smoke test: memory, speed, time estimate
One forward/backward pass at your batch size (same augmentation as the repo, BGR input like the repo).

In [ ]:
import time, torch, torch.nn as nn
from torch.utils.data import DataLoader

model = build_model(REPO, None, device='cuda', with_pretrained=True).train()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
ds_tr = PST900Dataset(PST_ROOT, 'train', is_training=True, crop_size=CROP, channel_order='BGR')
dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
it = iter(dl_tr)
torch.cuda.reset_peak_memory_stats(); times = []
for k in range(6):
    b = next(it, None)
    if b is None:
        it = iter(dl_tr); b = next(it)
    t0 = time.time()
    loss = model(b['rgb'].cuda(), b['thermal'].cuda(), b['label'].cuda())
    opt.zero_grad(); loss.backward(); opt.step(); torch.cuda.synchronize()
    if k: times.append(time.time() - t0)
    print(f'iter {k}: loss {loss.item():.3f}')
per_it = sum(times) / len(times)
iters_ep = counts['train'] // BATCH_SIZE + 1
print(f'\nPeak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
print(f'{per_it:.2f} s/iter -> {per_it*iters_ep/60:.1f} min/epoch -> ~{per_it*iters_ep*NEPOCHS/3600:.1f} h for {NEPOCHS} epochs')
print('Colab sessions are limited, so training will span several sessions: resume is automatic.')
del model, opt; torch.cuda.empty_cache()

### 12b. (Optional) Overfit sanity check
Train on 8 images only. Training mIoU should climb well above 80%.
If it does not, something is wrong with labels or the pipeline. Fix that before spending hours on full training.

In [ ]:
RUN_OVERFIT = False
if RUN_OVERFIT:
    from torch.utils.data import Subset
    model = build_model(REPO, None, device='cuda', with_pretrained=True).train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    sub = Subset(PST900Dataset(PST_ROOT, 'train', is_training=True, crop_size=CROP, scales=(1.0,), channel_order='BGR'), range(8))
    dl = DataLoader(sub, batch_size=4, shuffle=True)
    for ep in range(60):
        for b in dl:
            loss = model(b['rgb'].cuda(), b['thermal'].cuda(), b['label'].cuda())
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % 10 == 9:
            m = SegMetrics(5, PST900_CLASSES); model.eval()
            with torch.no_grad():
                for b in dl:
                    m.update(model(b['rgb'].cuda(), b['thermal'].cuda()).argmax(1), b['label'])
            model.train(); print(f'epoch {ep+1}: loss {loss.item():.3f}  train mIoU {m.compute()["mIoU"]*100:.1f}')
    del model, opt; torch.cuda.empty_cache()

## 13. TRAIN E2 (official `train.py` + CMRF, resumable)
* Same schedule as E1 (100 epochs). The training loss printed now includes the small CMRF prediction loss.
* If Colab disconnects, Run all again: it resumes from the newest checkpoint in Drive, and finished training is skipped.
* Console output is also saved to `LOG_DIR/train_stdout.txt`.

In [ ]:
import re
EXPECT_FINISHED_RUN = False    # E2 is a new run: training starts (or resumes) here

ck = latest_checkpoint(LOG_DIR)
done = int(re.search(r'epoch-(\d+)\.pth$', ck).group(1)) if ck else 0
print('Latest checkpoint:', ck if ck else 'none', f'(epoch {done}/{NEPOCHS})')

if ck is None and EXPECT_FINISHED_RUN:
    raise RuntimeError(
        f'No checkpoint found in {LOG_DIR}.\n'
        'Training was done with another Google account, or EXP was changed. Stopping so it does not retrain.\n'
        'Use the same Google account and EXP as the training run. To train from scratch on purpose, set EXPECT_FINISHED_RUN = False.')

if done >= NEPOCHS:
    print('Training already complete: skipping training and going to evaluation.')
else:
    resume = f'-c {ck}' if ck else ''
    print('Resuming from', ck if ck else '(scratch)')
    !cd {REPO} && TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python train.py -d 0 {resume} 2>&1 | tee -a {LOG_DIR}/train_stdout.txt

## 13b. Save the trained model
Saves the final weights only (about 270 MB, without the optimizer state) to `MyDrive/AIProj_CMX/models/` and downloads a copy.

In [ ]:
import torch
ck = latest_checkpoint(LOG_DIR)
assert ck, 'No checkpoint found: run the TRAIN cell first.'
state = torch.load(ck, map_location='cpu', weights_only=False)
done = state.get('epoch')
if done is not None and done < NEPOCHS:
    print(f'WARNING: training stopped at epoch {done}/{NEPOCHS}. Run the TRAIN cell again to finish first.')

MODEL_DIR = f'{PROJECT}/models'
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = f'{MODEL_DIR}/{EXP}_final.pth'
torch.save({'model': state['model'], 'epoch': done, 'exp': EXP, 'backbone': 'mit_b2',
            'num_classes': 5, 'classes': PST900_CLASSES, 'nepochs': NEPOCHS,
            'batch_size': BATCH_SIZE, 'lr': LR, 'crop': CROP, 'cmrf': CMRF_CFG}, MODEL_PATH)
print(f'Saved model (epoch {done}/{NEPOCHS}) -> {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB)')

# check that the saved file loads correctly
_m = build_model(REPO, MODEL_PATH, device='cpu'); del _m
print('Reload check passed.')

try:                                   # download a copy
    from google.colab import files
    files.download(MODEL_PATH)
except Exception as e:
    print('Automatic download skipped:', e, '| the file is in Google Drive:', MODEL_PATH)

## 14. Evaluate E1: clean + corruption benchmark
* Sliding-window inference with 480×640 windows and stride 2/3, scored on **full-resolution 720×1280 labels** (same protocol as the repo's evaluator).
* `channel_order='BGR'`: the official loader feeds images in BGR order (its `imread` flag is a colour-conversion constant), so evaluation must match.
* Results are appended to `RESULTS/benchmark.csv` with the experiment tag, so E1–E4 end up in one table.

In [ ]:
import pandas as pd
ck = latest_checkpoint(LOG_DIR); print('Evaluating', ck)
model = build_model(REPO, ck, device='cuda')

rows = run_benchmark(model, PST_ROOT, crop=CROP, stride_rate=2/3, flip=False, channel_order='BGR',
                     out_csv=f'{RESULTS}/benchmark.csv', tag=EXP)
summary = robustness_summary(rows)
pd.Series(summary).mul(100).round(2).to_frame('%')

In [ ]:
# Per-class table on clean data (the 'failure cases' discussion starts here)
clean = rows[0]
per_class = pd.DataFrame({m: [clean[f'{m}/{c}']*100 for c in PST900_CLASSES]
                          for m in ('IoU', 'Dice', 'Sensitivity', 'Specificity')}, index=PST900_CLASSES).round(2)
per_class.loc['mean (all)'] = [clean[k]*100 for k in ('mIoU', 'mDice', 'mSensitivity', 'mSpecificity')]
per_class.loc['mean (objects)'] = [clean[k]*100 for k in ('mIoU_obj', 'mDice_obj', 'mSensitivity_obj', 'mSpecificity_obj')]
per_class.to_csv(f'{RESULTS}/{EXP}_per_class_clean.csv')
per_class.round(2)

In [ ]:
# Robustness table: mIoU per corruption x severity
df = pd.DataFrame(rows)
tab = df[df.condition != 'clean'].pivot(index='condition', columns='severity', values='mIoU').mul(100).round(2)
tab['avg'] = tab.mean(1).round(2)
tab.loc['clean (reference)'] = round(rows[0]['mIoU']*100, 2)
tab.to_csv(f'{RESULTS}/{EXP}_robustness_mIoU.csv')
tab

## 15. Cost: parameters, GFLOPs, latency (single 480×640 input)

In [ ]:
cost = model_cost(model, input_size=CROP)
pd.Series(cost).to_frame(EXP)

## 15b. E1 vs E2 comparison
E1 reference numbers are taken from the E1 run (100 epochs, `E1_cmx_b2_ep300` results). Positive Δ means E2 is better (for latency, lower is better).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
E1 = {'clean': {'mIoU': 86.12, 'mIoU_obj': 82.76, 'mDice': 92.37, 'mSensitivity': 90.96, 'mSpecificity': 97.91,
                'IoU/Background': 99.57, 'IoU/Fire-Extinguisher': 80.43, 'IoU/Backpack': 88.35,
                'IoU/Hand-Drill': 83.46, 'IoU/Survivor': 78.78},
      'robust': {'rgb_blur': [82.29, 71.43, 49.63], 'rgb_low_light': [78.19, 42.71, 19.57],
                 'rgb_noise': [85.48, 83.11, 77.86], 'rgb_occlusion': [80.35, 74.09, 64.32],
                 'thermal_blur': [86.04, 85.95, 85.81], 'thermal_noise': [85.55, 85.02, 84.78],
                 'thermal_occlusion': [85.71, 85.62, 85.32]},
      'cost': {'params_M': 66.56, 'GFLOPs': 78.27, 'latency_ms': 109.79}}

clean_e2 = rows[0]
t1 = pd.DataFrame({'E1': E1['clean'], 'E2': {k: clean_e2[k] * 100 for k in E1['clean']}})
t1['Δ (E2−E1)'] = t1.E2 - t1.E1
print('Clean test set'); display(t1.round(2))

df2 = pd.DataFrame(rows)
t2 = []
for cond, vals in E1['robust'].items():
    for s, v1 in zip((1, 2, 3), vals):
        v2 = df2[(df2.condition == cond) & (df2.severity == s)].mIoU.values[0] * 100
        t2.append({'condition': cond, 'severity': s, 'E1': v1, 'E2': v2, 'Δ': v2 - v1})
t2 = pd.DataFrame(t2)
summary = pd.DataFrame({
    'E1': [E1['clean']['mIoU'], t2[t2.condition.str.startswith('rgb')].E1.mean(), t2[t2.condition.str.startswith('thermal')].E1.mean(),
           E1['robust']['rgb_low_light'][2]],
    'E2': [clean_e2['mIoU'] * 100, t2[t2.condition.str.startswith('rgb')].E2.mean(), t2[t2.condition.str.startswith('thermal')].E2.mean(),
           t2[(t2.condition == 'rgb_low_light') & (t2.severity == 3)].E2.values[0]]},
    index=['clean mIoU', 'avg under RGB degradation', 'avg under thermal degradation', 'RGB low-light s3'])
summary['Δ'] = summary.E2 - summary.E1
print('\nRobustness summary'); display(summary.round(2))
print('\nPer condition'); display(t2.round(2))

c2 = cost
tc = pd.DataFrame({'E1': E1['cost'], 'E2': {'params_M': c2['params_M'], 'GFLOPs': c2['GFLOPs'] if not isinstance(c2['GFLOPs'], str) else np.nan,
                                            'latency_ms': c2['latency_ms']}})
tc['overhead %'] = 100 * (tc.E2 - tc.E1) / tc.E1
print('\nCost (latency depends on the GPU; compare on the same GPU)'); display(tc.round(2))

t1.round(2).to_csv(f'{RESULTS}/E1_vs_E2_clean.csv'); t2.round(2).to_csv(f'{RESULTS}/E1_vs_E2_robustness.csv', index=False)
summary.round(2).to_csv(f'{RESULTS}/E1_vs_E2_summary.csv'); tc.round(2).to_csv(f'{RESULTS}/E1_vs_E2_cost.csv')

fig, axes = plt.subplots(1, 7, figsize=(24, 3.6), sharey=True)
for ax, cond in zip(axes, E1['robust']):
    sub = t2[t2.condition == cond]
    ax.plot([0, 1, 2, 3], [E1['clean']['mIoU']] + list(sub.E1), 'o-', label='E1 (CMX)')
    ax.plot([0, 1, 2, 3], [clean_e2['mIoU'] * 100] + list(sub.E2), 's-', label='E2 (CMX+CMRF)')
    ax.set_title(cond); ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(['clean', 's1', 's2', 's3']); ax.set_ylim(0, 100)
axes[0].set_ylabel('mIoU (%)'); axes[0].legend()
plt.tight_layout(); plt.savefig(f'{RESULTS}/E1_vs_E2_robustness.png', dpi=120); plt.show()

## 16. Qualitative results under degradation
This shows where the model fails, which is the evidence for CMRF.
It is especially useful under `rgb_low_light` (does thermal rescue it?) and `thermal_occlusion` (does it break?).

In [ ]:
import matplotlib.pyplot as plt
SHOW = [('clean', 1), ('rgb_low_light', 3), ('rgb_occlusion', 3), ('thermal_occlusion', 3)]
idxs = [3, 100, 200]
fig, ax = plt.subplots(len(idxs), 2 + len(SHOW), figsize=(4*(2+len(SHOW)), 3*len(idxs)))
for r, i in enumerate(idxs):
    for c, (cn, sev) in enumerate(SHOW):
        d = PST900Dataset(PST_ROOT, 'test', corruption=cn, severity=sev, channel_order='BGR')
        b = d[i]
        p = predict_probs(model, b['rgb'][None].cuda(), b['thermal'][None].cuda(), crop=CROP).argmax(1)[0].cpu().numpy()
        if c == 0:
            raw = d.load_raw(i)
            ax[r, 0].imshow(raw[0]); ax[r, 0].set_title(raw[3])
            ax[r, 1].imshow(colorize(raw[2])); ax[r, 1].set_title('Ground truth')
        ax[r, 2+c].imshow(colorize(p)); ax[r, 2+c].set_title(f'{cn} s{sev}')
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.savefig(f'{RESULTS}/{EXP}_qualitative.png', dpi=120); plt.show()

## 16b. Modality-drop test
Evaluates E2 with thermal blanked out (RGB only) and with RGB blanked out (thermal only).
Run the same cell on the E1 model for the matching E1 numbers.

In [ ]:
import torch
drop_rows = []
for tag, fwd in [(f'{EXP}_rgb_only',     lambda m, r, t: m(r, torch.zeros_like(t))),
                 (f'{EXP}_thermal_only', lambda m, r, t: m(torch.zeros_like(r), t))]:
    r = run_benchmark(model, PST_ROOT, forward_fn=fwd, corruptions=[], crop=CROP, channel_order='BGR',
                      out_csv=f'{RESULTS}/modality_drop.csv', tag=tag)[0]
    drop_rows.append({'input': tag.split('_')[-2] + ' only', 'mIoU': r['mIoU'] * 100, 'mIoU_obj': r['mIoU_obj'] * 100})
drop_rows.insert(0, {'input': 'RGB + thermal', 'mIoU': rows[0]['mIoU'] * 100, 'mIoU_obj': rows[0]['mIoU_obj'] * 100})
pd.DataFrame(drop_rows).round(2)

## 16c. What CMRF does: residual and gate maps
For the same test images, under **clean** and under **RGB low-light (severity 3)**:
* **thermal residual** |F_thm − P(F_rgb)|: thermal information RGB cannot explain;
* **thermal→RGB gate**: how much of it CMRF injects into the RGB stream.

If the hypothesis holds, the gate and residual should become stronger on objects (for example survivors) when RGB is dark.
The maps are from a single 480×640 forward pass (resized input), at the CMRF stage's resolution.

In [ ]:
import cv2, torch, torch.nn.functional as F
cm = model.backbone.CMRFs[str(CMRF_STAGES[0])]
idxs_g = [3, 100]
fig, ax = plt.subplots(len(idxs_g), 6, figsize=(24, 3.4 * len(idxs_g)))
for r, i in enumerate(idxs_g):
    for c, (cn, sev) in enumerate([('clean', 1), ('rgb_low_light', 3)]):
        d = PST900Dataset(PST_ROOT, 'test', corruption=cn, severity=sev, channel_order='BGR')
        b = d[i]
        rgb = F.interpolate(b['rgb'][None], size=CROP, mode='bilinear', align_corners=False).cuda()
        th = F.interpolate(b['thermal'][None], size=CROP, mode='bilinear', align_corners=False).cuda()
        with torch.no_grad():
            model.eval()(rgb, th)
        res = cm.last['res_t'][0].cpu().numpy(); gate = cm.last['gate_t'][0].cpu().numpy()
        img = d.load_raw(i)[0] if cn == 'clean' else None
        if cn == 'clean':
            ax[r, 0].imshow(img); ax[r, 0].set_title(f'RGB (clean) {d.load_raw(i)[3][:20]}')
        else:
            dark = (b['rgb'].permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)[..., ::-1]
            ax[r, 3].imshow(dark); ax[r, 3].set_title('RGB (low-light s3)')
        ax[r, 1 + 3 * c].imshow(res, cmap='magma'); ax[r, 1 + 3 * c].set_title(f'thermal residual ({cn})')
        ax[r, 2 + 3 * c].imshow(gate, cmap='viridis', vmin=0, vmax=1); ax[r, 2 + 3 * c].set_title(f'thermal→RGB gate ({cn})')
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.savefig(f'{RESULTS}/{EXP}_cmrf_maps.png', dpi=120); plt.show()
print('learned injection strength: scale_t =', round(cm.scale_t.item(), 4), '| scale_r =', round(cm.scale_r.item(), 4))

## 17. How to read the E2 results
1. **Clean accuracy:** E2 should stay about equal to E1 (86.12). A small drop is acceptable; a large drop means CMRF hurts.
2. **Robustness (main question):** compare the *average under RGB degradation* (E1: 67.42) and *RGB low-light s3* (E1: 19.57).
3. **Thermal use:** if CMRF works, damaging thermal should now matter a little more, because the model uses it.
4. **Cost:** expected +0.70% params and +0.71% GFLOPs.
5. **Gate maps:** the gate should be stronger on objects when RGB is dark.

E2 alone is trained on **clean** images only. The bigger robustness gain is expected in **E3** (corruption during training), which needs the **control run** (E1 + the same corruption, without CMRF) to show CMRF's own contribution.

## 18. Package results to send back
Creates `E2_results.zip` in `MyDrive/AIProj_CMX/` (all CSVs, E1 vs E2 tables and figures, CMRF maps, training log) and downloads it.

In [ ]:
import shutil, glob, os
pkg = '/content/E2_results'
shutil.rmtree(pkg, ignore_errors=True); os.makedirs(pkg)
for f in glob.glob(f'{RESULTS}/*') + glob.glob(f'{LOG_DIR}/*.txt') + glob.glob(f'{LOG_DIR}/*.log'):
    if os.path.isfile(f): shutil.copy(f, pkg)
zip_path = shutil.make_archive(f'{PROJECT}/E2_results', 'zip', pkg)
print('Saved:', zip_path)
from google.colab import files
files.download(zip_path)